[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/24_rope.ipynb)

# 🔴 Hard: Rotary Position Embedding (RoPE)

Implement **RoPE** — the position encoding used in LLaMA, GPT-NeoX, and most modern LLMs.

### Signature
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D) where D is even
    # Returns rotated (q, k) with same shape
```

### Key Idea
Split each vector into consecutive pairs. Rotate each pair by `θ = pos / 10000^(2i/D)`:
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
This makes `dot(q_rot[i], k_rot[j])` depend only on `i - j` (relative position).

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.0 MB/s eta 0:00:00


In [2]:
import torch
import math

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

def apply_rope(q, k):
  B, Tq, Dq = q.shape
  B, Tk, Dk = k.shape
  i = torch.arange(Tq).unsqueeze(1)
  j = torch.arange(Tk).unsqueeze(1)
  p_idx_q = torch.arange(Dq//2).unsqueeze(0)
  p_idx_k = torch.arange(Dk//2).unsqueeze(0)
  theta_q = i/torch.pow(10000, (2*p_idx_q)/Dq)
  theta_k = j/torch.pow(10000, (2*p_idx_k)/Dk)

  q_even = q[..., 0::2]
  q_odd = q[..., 1::2]
  k_even = k[..., 0::2]
  k_odd = k[..., 1::2]

  re_q = q_even * torch.cos(theta_q) - q_odd * torch.sin(theta_q)
  ro_q = q_even * torch.sin(theta_q) + q_odd * torch.cos(theta_q)
  re_k = k_even * torch.cos(theta_k) - k_odd * torch.sin(theta_k)
  ro_k = k_even * torch.sin(theta_k) + k_odd * torch.cos(theta_k)

  r_q = torch.stack([re_q, ro_q], dim=-1).reshape(B, Tq, Dq)
  r_k = torch.stack([re_k, ro_k], dim=-1).reshape(B, Tk, Dk)

  return r_q, r_k

    # 1. Compute position angles
    # 2. Split into even/odd pairs
    # 3. Apply rotation
    # pass

In [4]:
# 🧪 Debug
q = torch.randn(1, 8, 16)
k = torch.randn(1, 8, 16)
qr, kr = apply_rope(q, k)
print('Shape preserved:', qr.shape == q.shape)
print('Norm preserved:', torch.allclose(q.norm(dim=-1), qr.norm(dim=-1), atol=1e-4))

Shape preserved: True
Norm preserved: True


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('rope')


🧪 Testing: Rotary Position Embedding (RoPE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shapes (3.5ms)
  ✅ [2/4] Preserves norm (4.7ms)
  ✅ [3/4] Relative position property (17.3ms)
  ✅ [4/4] Gradient flow (17.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (42.7ms total)
  Progress saved. Run status() to see your dashboard.

